This notebook creates the TrafficPerTerritory table the first time. Once created and loaded in the data warehouse it will be updated by another
notebook or script that will only retrieve new data from the API

In [57]:
import pandas as pd
import utils as u

In [58]:
# url passenger data
url_pas = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000013/~latest.csv?lang=en"
# url goods and mail data
url_gm = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000014/~latest.csv?lang=en"
# url operations
url_o = "https://datos.canarias.es/api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000015/~latest.csv?lang=en"

In [59]:
# Last updated passenger data
passg = pd.read_csv(u.get_data_from_API_call(url_pas))
# Last updated goods and mail data
goodsm = pd.read_csv(u.get_data_from_API_call(url_gm))
# Last updated operations data
operat = pd.read_csv(u.get_data_from_API_call(url_o))

ConnectionError: HTTPSConnectionPool(host='datos.canarias.es', port=443): Max retries exceeded with url: /api/estadisticas/statistical-resources/v1.0/datasets/ISTAC/C00017A_000013/~latest.csv?lang=en (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x799cddd41c40>: Failed to establish a new connection: [Errno 113] No route to host'))

In [ ]:
passg

,MEDIDAS#en,MEDIDAS#es,MEDIDAS_CODE,TERRITORIO#en,TERRITORIO#es,TERRITORIO_CODE,AEROPUERTO_ESCALA#en,AEROPUERTO_ESCALA#es,AEROPUERTO_ESCALA_CODE,MOVIMIENTO_AERONAVE#en,...,SERVICIO_AEREO_CODE,TIME_PERIOD#en,TIME_PERIOD#es,TIME_PERIOD_CODE,OBS_VALUE,ESTADO_OBSERVACION#en,ESTADO_OBSERVACION#es,ESTADO_OBSERVACION_CODE,CONFIDENCIALIDAD_OBSERVACION#en,CONFIDENCIALIDAD_OBSERVACION#es
0,Passengers,Pasajeros,PASAJEROS,Canary Islands,Canarias,ES70,United Kingdom of Great Britain and Northern I...,Reino Unido,GB,Arrival,...,COMMERCIAL,01/2004,01/2004,2004-M01,317378.0,Normal value,Valor normal,A,NaN,NaN
1,Passengers,Pasajeros,PASAJEROS,Canary Islands,Canarias,ES70,United Kingdom of Great Britain and Northern I...,Reino Unido,GB,Arrival,...,COMMERCIAL,02/2004,02/2004,2004-M02,320960.0,Normal value,Valor normal,A,NaN,NaN
2,Passengers,Pasajeros,PASAJEROS,Canary Islands,Canarias,ES70,United Kingdom of Great Britain and Northern I...,Reino Unido,GB,Arrival,...,COMMERCIAL,06/2010,06/2010,2010-M06,251362.0,Normal value,Valor normal,A,NaN,NaN
3,Passengers,Pasajeros,PASAJEROS,Canary Islands,Canarias,ES70,United Kingdom of Great Britain and Northern I...,Reino Unido,GB,Arrival,...,COMMERCIAL,07/2010,07/2010,2010-M07,296367.0,Normal value,Valor normal,A,NaN,NaN
4,Passengers,Pasajeros,PASAJEROS,Canary Islands,Canarias,ES70,United Kingdom of Great Britain and Northern I...,Reino Unido,GB,Arrival,...,COMMERCIAL,08/2010,08/2010,2010-M08,287040.0,Normal value,Valor normal,A,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215035,Passengers,Pasajeros,PASAJEROS,El Hierro,El Hierro,ES703,Germany,Alemania,DE,Total,...,SCHEDULED,01/2010,01/2010,2010-M01,NaN,NaN,NaN,NaN,NaN,NaN
215036,Passengers,Pasajeros,PASAJEROS,El Hierro,El Hierro,ES703,Germany,Alemania,DE,Total,...,SCHEDULED,02/2010,02/2010,2010-M02,NaN,NaN,NaN,NaN,NaN,NaN
215037,Passengers,Pasajeros,PASAJEROS,El Hierro,El Hierro,ES703,Germany,Alemania,DE,Total,...,SCHEDULED,03/2010,03/2010,2010-M03,NaN,NaN,NaN,NaN,NaN,NaN
215038,Passengers,Pasajeros,PASAJEROS,El Hierro,El Hierro,ES703,Germany,Alemania,DE,Total,...,SCHEDULED,04/2010,04/2010,2010-M04,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# For some reason the operations table changes the name of the column of the stopover airport
operat.rename({'AEROPUERTO_ORIGEN_DESTINO_CODE': 'AEROPUERTO_ESCALA_CODE', 'AEROPUERTO_ORIGEN_DESTINO#en': 'AEROPUERTO_ESCALA#en'}, axis=1, inplace=True)

In [ ]:
dfs = [passg, goodsm, operat]

In [ ]:
# Delete unnecesary columns
for df in dfs:
    df.drop(columns=df.columns[df.columns.str.endswith("#es")], inplace=True)
    df.drop(columns=['CONFIDENCIALIDAD_OBSERVACION#en', 'ESTADO_OBSERVACION_CODE', 'ESTADO_OBSERVACION#en'], inplace=True)

In [ ]:
# Merge passengers and operations
df_f = passg.merge(operat, on=['TERRITORIO_CODE', 'AEROPUERTO_ESCALA_CODE', 'MOVIMIENTO_AERONAVE_CODE', 'SERVICIO_AEREO_CODE', 'TIME_PERIOD_CODE'], suffixes=("", "_op"))

df_f.drop(columns=[col for col in df_f.columns if col.endswith('op') and not col.endswith('_VALUE_op')], inplace=True)

In [ ]:
only_goods = goodsm.loc[goodsm['MEDIDAS_CODE'] == "MERCANCIA"].copy(deep=True)

only_mail = goodsm.loc[goodsm['MEDIDAS_CODE'] == "CORREO"].copy(deep=True)

In [ ]:
# Merge p&op with goods 
df_f = df_f.merge(only_goods, on=['TERRITORIO_CODE', 'AEROPUERTO_ESCALA_CODE', 'MOVIMIENTO_AERONAVE_CODE', 'SERVICIO_AEREO_CODE', 'TIME_PERIOD_CODE'], suffixes=("", "_goods"))

df_f.drop(columns=[col for col in df_f.columns if col.endswith('goods') and col not in ['OBS_VALUE_goods']], inplace=True)

# Merge p&op&goods with mail 
df_f = df_f.merge(only_mail, on=['TERRITORIO_CODE', 'AEROPUERTO_ESCALA_CODE', 'MOVIMIENTO_AERONAVE_CODE', 'SERVICIO_AEREO_CODE', 'TIME_PERIOD_CODE'], suffixes=("", "_mail"))

df_f.drop(columns=[col for col in df_f.columns if col.endswith('mail') and col not in ['OBS_VALUE_mail']], inplace=True)

Only columns with null values are the OBS_VALUE, this is because the ISTAC dataset treats observed values with value 0 as nan/null

In [ ]:
df_f.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215040 entries, 0 to 215039
Data columns (total 16 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   MEDIDAS#en                215040 non-null  object 
 1   MEDIDAS_CODE              215040 non-null  object 
 2   TERRITORIO#en             215040 non-null  object 
 3   TERRITORIO_CODE           215040 non-null  object 
 4   AEROPUERTO_ESCALA#en      215040 non-null  object 
 5   AEROPUERTO_ESCALA_CODE    215040 non-null  object 
 6   MOVIMIENTO_AERONAVE#en    215040 non-null  object 
 7   MOVIMIENTO_AERONAVE_CODE  215040 non-null  object 
 8   SERVICIO_AEREO#en         215040 non-null  object 
 9   SERVICIO_AEREO_CODE       215040 non-null  object 
 10  TIME_PERIOD#en            215040 non-null  object 
 11  TIME_PERIOD_CODE          215040 non-null  object 
 12  OBS_VALUE                 143567 non-null  float64
 13  OBS_VALUE_op              167570 non-null  f

In [ ]:
# nan values mean the observed value is 0
df_f.fillna(0, inplace=True)

Change columns to Id's, change name of columns

In [ ]:
df_f.drop(columns=[col for col in df_f.columns if col.endswith('#en') and col not in ['TIME_PERIOD#en']], inplace=True)
df_f.drop(columns=['TIME_PERIOD_CODE', 'MEDIDAS_CODE'], inplace=True)

In [ ]:
territory = pd.read_csv('../../data/Territory.csv')
airservice = pd.read_csv('../../data/AirService.csv')
aircraftmovement = pd.read_csv('../../data/AircraftMovement.csv')

In [ ]:
df_f.loc[pd.isna(df_f["AEROPUERTO_ESCALA_CODE"])]

,TERRITORIO_CODE,AEROPUERTO_ESCALA_CODE,MOVIMIENTO_AERONAVE_CODE,SERVICIO_AEREO_CODE,TIME_PERIOD#en,OBS_VALUE,OBS_VALUE_op,OBS_VALUE_goods,OBS_VALUE_mail


In [ ]:
# Replace values in df_f["TERRITORIO_CODE"] with corresponding TerritoryId
df_f["TERRITORIO_CODE"] = df_f["TERRITORIO_CODE"].map(dict(zip(territory["TerritoryCode"], territory["TerritoryId"])))

# Replace values in df_f["AEROPUERTO_ESCALA_CODE"] with corresponding TerritoryId
df_f["AEROPUERTO_ESCALA_CODE"] = df_f["AEROPUERTO_ESCALA_CODE"].map(dict(zip(territory["TerritoryCode"], territory["TerritoryId"])))
# Delete rows where AEROPUERTO_ESCALA_CODE hasnt matched (deleted territories like Total)
df_f.dropna(inplace=True)
df_f["AEROPUERTO_ESCALA_CODE"] = df_f["AEROPUERTO_ESCALA_CODE"].astype(int)

df_f["SERVICIO_AEREO_CODE"] = df_f["SERVICIO_AEREO_CODE"].map(dict(zip(airservice['AirServiceCode'], airservice['AirServiceId'])))

df_f["MOVIMIENTO_AERONAVE_CODE"] = df_f["MOVIMIENTO_AERONAVE_CODE"].map(dict(zip(aircraftmovement['AircraftMovementCode'], aircraftmovement['AircraftMovementId'])))

In [ ]:
# Delete year only dates
df_f = df_f.loc[~df_f['TIME_PERIOD#en'].str.match(r'^20[0-9][0-9]$', na=False)]

df_f['TIME_PERIOD#en'] = pd.to_datetime(df_f['TIME_PERIOD#en'], format='%m/%Y')

In [ ]:
df_f.sort_values(by='TIME_PERIOD#en', ascending=False)

,TERRITORIO_CODE,AEROPUERTO_ESCALA_CODE,MOVIMIENTO_AERONAVE_CODE,SERVICIO_AEREO_CODE,TIME_PERIOD#en,OBS_VALUE,OBS_VALUE_op,OBS_VALUE_goods,OBS_VALUE_mail
42198,6,0,0,0,2025-07-01,51789.0,985.0,16833.0,0.0
154758,7,8,0,2,2025-07-01,0.0,0.0,0.0,0.0
103238,7,0,2,1,2025-07-01,0.0,4.0,0.0,0.0
182198,3,8,2,3,2025-07-01,148632.0,851.0,0.0,0.0
212438,7,11,2,3,2025-07-01,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
143360,5,0,2,2,2004-01-01,0.0,0.0,0.0,0.0
144480,5,11,0,2,2004-01-01,0.0,0.0,0.0,0.0
144760,5,11,1,2,2004-01-01,0.0,0.0,0.0,0.0
145040,5,11,2,2,2004-01-01,0.0,0.0,0.0,0.0


If everything was done right, the number of rows should be equal to number of months from 01/2004 to 07/2025 * len(territorio_code.unique) * len(aeropuerto_escala_code.unique) * (movimiento_aeronave_code.unique) * (servicio_aereo_code.unique)

In [ ]:
len(df_f['TIME_PERIOD#en'].unique()) * (len(df_f['TERRITORIO_CODE'].unique())) * (len(df_f['AEROPUERTO_ESCALA_CODE'].unique())) * \
(len(df_f['MOVIMIENTO_AERONAVE_CODE'].unique())) * (len(df_f['SERVICIO_AEREO_CODE'].unique())) == df_f.shape[0]

True

In [ ]:
df_f.rename({
'TERRITORIO_CODE': 'IslandId',
'AEROPUERTO_ESCALA_CODE': 'StopoverTerritoryId',
'MOVIMIENTO_AERONAVE_CODE': 'AircraftMovementId',
'SERVICIO_AEREO_CODE': 'AirServiceId',
'TIME_PERIOD#en': 'Month',
'OBS_VALUE': 'Passengers',
'OBS_VALUE_op': 'Operations',
'OBS_VALUE_goods': 'Goods',
'OBS_VALUE_mail': 'Mail'
}, inplace=True, axis=1)

In [ ]:
for col in ['Passengers', 'Goods', 'Mail', 'Operations']:
    df_f[col] = df_f[col].astype(int)

Delete Germany and Uk passengers from FOREIGN passengers

In [ ]:
# Make a copy to work with
df_copy = df_f.copy()

# Get all rows where StopoverTerritoryId == 9 (FOREIGN)
foreign_mask = df_f['StopoverTerritoryId'] == 9
foreign_rows = df_f[foreign_mask].copy()

# Define the measure columns to subtract from
measure_cols = ['Passengers', 'Operations', 'Goods', 'Mail']

# For each foreign row, find and subtract corresponding GERMANY (14) and UK (8) rows
for idx in foreign_rows.index:
    # Get the identifying values for this row
    island_id = df_f.loc[idx, 'IslandId']
    aircraft_id = df_f.loc[idx, 'AircraftMovementId']
    airservice_id = df_f.loc[idx, 'AirServiceId']
    month = df_f.loc[idx, 'Month']
    
    # Find corresponding GERMANY row (StopoverTerritoryId == 14)
    germany_mask = (
        (df_f['IslandId'] == island_id) &
        (df_f['AircraftMovementId'] == aircraft_id) &
        (df_f['AirServiceId'] == airservice_id) &
        (df_f['Month'] == month) &
        (df_f['StopoverTerritoryId'] == 14)
    )
    
    # Find corresponding UK row (StopoverTerritoryId == 8)
    uk_mask = (
        (df_f['IslandId'] == island_id) &
        (df_f['AircraftMovementId'] == aircraft_id) &
        (df_f['AirServiceId'] == airservice_id) &
        (df_f['Month'] == month) &
        (df_f['StopoverTerritoryId'] == 8)
    )
    
    # Initialize subtraction values
    subtract_values = {col: 0 for col in measure_cols}
    
    # Add values from GERMANY row if it exists
    if germany_mask.any():
        germany_idx = df_f[germany_mask].index[0]
        for col in measure_cols:
            subtract_values[col] += df_f.loc[germany_idx, col]
    
    # Add values from UK row if it exists
    if uk_mask.any():
        uk_idx = df_f[uk_mask].index[0]
        for col in measure_cols:
            subtract_values[col] += df_f.loc[uk_idx, col]
    
    # Subtract from the FOREIGN row
    for col in measure_cols:
        df_copy.loc[idx, col] = df_f.loc[idx, col] - subtract_values[col]

# Update the original dataframe
df_f.update(df_copy)

In [ ]:
df_f.sort_values(by=['IslandId', 'Month']).to_csv('../../data/TrafficPerTerritory.csv', index=False)